# Demo 2: The graph proposes, Jev decides, the graph remembers

In Demo 1 we described two branches in a few words and asked Jev which to take. Here Neo4j does more of the work:

1. **The graph proposes** the candidate routes from Kennington to Euston (Graph Data Science, Yen's k shortest paths).
2. **Cypher adds the live status** of each route, read from the station properties.
3. **Jev decides** with `jev.decide` and says how sure it is.
4. **The graph remembers.** Each decision is stored as a node linked to the stations that were disrupted, so we can ask questions about it afterwards.

- Run the cells one at a time, from the top.
- Hosted Jev is the default. The optional section near the end runs the same call on a local model.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="your-database"
```

You also need:

- The Jev plugin JAR and the Graph Data Science plugin (both from Neo4j Desktop's plugins folder), with `dbms.security.procedures.allowlist=jev.*,gds.*` and `dbms.security.procedures.unrestricted=gds.*` in `neo4j.conf`.
- For the hosted API, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.
- The three London Underground CSV files in the shared `data` folder, one level above this notebook's folder, the same ones the Streamlit app uses. Section 5 clears the target database and loads them, so point `NEO4J_DATABASE` at a database you're happy to wipe.
- For the local model (optional), Ollama running with the model pulled (`ollama pull tev1:4b`).

## 1. Install Dependencies

In [1]:
%pip install neo4j==6.4.1 \
             pandas==3.1.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import re
import pandas as pd

from IPython.display import display
from neo4j import GraphDatabase, NotificationMinimumSeverity
from neo4j.exceptions import AuthError, ServiceUnavailable, SessionExpired

## 3. Configuration

The connection details come from environment variables set before Jupyter was launched.

The journey is set here, so you can change the trip without touching any query. `LINES` limits which lines the graph may use (`None` means any line) and `K` is how many routes it proposes. The graph finds a pool of `POOL` short routes, ranks them by stops plus `INTERCHANGE_COST` for each interchange, and keeps the `K` best that share at most `MAX_OVERLAP` of their stations, so the options really are different. On the Northern line from Kennington to Euston the two shortest routes are the two branches.

`HOSTED_OPTIONS` and `LOCAL_OPTIONS` are the third argument of `jev.decide`. A short timeout and few retries make a failing call come back quickly, as an `error_message` in the result instead of a stack trace. The hosted model is pinned to the version Demo 1 uses.

`DEBUG` is off by default, so a failure prints one line that says what to do and the notebook carries on. Turn it on to see the full error while you're developing.

In [3]:
DEBUG = False   # True: show full tracebacks instead of one-line messages

SETTINGS = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]
MISSING  = [name for name in SETTINGS if not os.environ.get(name)]

NEO4J_URI      = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE")

ORIGIN      = "Kennington"
DESTINATION = "Euston"
LINES       = ["Northern"]   # None: any line
K           = 2              # how many routes the graph proposes
POOL        = 40             # how many short routes Yen's finds before the distinct ones are picked
MAX_OVERLAP = 0.5            # two routes may share at most this fraction of their stations
INTERCHANGE_COST = 3         # an interchange counts as this many extra stops when routes are ranked
GRAPH_NAME  = "routes"       # the name of the GDS projection

CLEAR_DATABASE = True    # True: section 5 deletes everything in the target database first. Point it at a database you're happy to wipe
RELOAD_DATA    = False   # True: delete the Station nodes and load them again

HOSTED_OPTIONS = {"model": "jev-1.13.0", "max_retries": 1, "request_timeout_ms": 8000}   # pinned, same model as Demo 1

LOCAL_OPTIONS = {
    "url":   "http://localhost:11434/v1/systemone",
    "model": "tev1:4b",
    "max_retries": 0,
    "request_timeout_ms": 60000,
}

if MISSING:
    print(f"Not set: {', '.join(MISSING)}. Set them before launching Jupyter (see the prerequisites), then restart the kernel.")
else:
    print("Credentials set.")

Credentials set.


### Error handling

Whatever goes wrong (Neo4j isn't running, the login is wrong, a plugin is missing), the cells print one line that says what to do and then stop, with no traceback. Once the problem is fixed, run the cell that failed again. You don't need to restart the kernel.

In [4]:
class NotConnected(Exception):
    pass

driver = None

def explain(exc):
    # Turn an error into one sentence that says what to do.
    text = str(exc)
    code = getattr(exc, "code", "") or ""
    if isinstance(exc, NotConnected):
        if MISSING:
            return f"Not connected. Not set: {', '.join(MISSING)}. Set them before launching Jupyter, then restart the kernel."
        return "Not connected to Neo4j. Run section 4 first."
    if isinstance(exc, AuthError):
        return "Neo4j rejected the login. Check NEO4J_USERNAME and NEO4J_PASSWORD, then relaunch Jupyter."
    if isinstance(exc, (ServiceUnavailable, SessionExpired, ConnectionError, OSError)):
        return f"Can't reach Neo4j at {NEO4J_URI}. Start the DBMS in Neo4j Desktop, then run this cell again."
    if "DatabaseNotFound" in code:
        return "Database '" + str(NEO4J_DATABASE) + "' doesn't exist. Check NEO4J_DATABASE."
    if "DatabaseUnavailable" in code:
        return "The database isn't available yet. Wait for it to finish starting, then run this cell again."
    if "jev." in text and ("Unknown function" in text or "ProcedureNotFound" in code or "no procedure" in text.lower()):
        return "The jev functions aren't available. Check the JAR, the allowlist line in neo4j.conf and that the DBMS was restarted."
    if "gds." in text and ("Unknown function" in text or "ProcedureNotFound" in code or "no procedure" in text.lower()
                           or "not allowed" in text.lower() or "permission" in text.lower()):
        return "The gds procedures aren't available. Install the Graph Data Science plugin, add gds.* to the allowlist and unrestricted lines in neo4j.conf and restart the DBMS."
    return f"The query failed: {type(exc).__name__}: {text[:300]}"

def problem(exc):
    # Print one line. With DEBUG on, raise the original error instead.
    print(explain(exc))
    if DEBUG:
        raise exc

def open_session():
    if driver is None:
        raise NotConnected()
    return driver.session(database=NEO4J_DATABASE)

def describe_error(message):
    # Say in plain words what went wrong with a Jev call, from the error_message in its result.
    lower = message.lower()
    if "connectexception" in lower or "unknownhost" in lower or "timeout" in lower or "timed out" in lower:
        if "11434" in message or "localhost" in lower:
            return "Can't reach the local model (is Ollama running?)."
        return "Can't reach the hosted Jev (check the network)."
    return message[:200]

print("Error handling defined.")

Error handling defined.


## 4. Connect to Neo4j

In [5]:
if driver is not None:
    driver.close()
driver = None

if MISSING:
    print(explain(NotConnected()))
else:
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
        notifications_min_severity = NotificationMinimumSeverity.OFF,
    )
    try:
        driver.verify_connectivity()
        print("Connection created.")
    except Exception as exc:
        problem(exc)

Connection created.


## 5. Clear the database and load the network

The first cell deletes every node, relationship, constraint and index in the target database so the notebook is safe to re-run. Set `CLEAR_DATABASE = False` in section 3 to skip it. The next cells read the three CSV files and load the whole London Underground network. Each station becomes a `Station` node with coordinates, and each connection becomes a pair of `CONNECTS_TO` relationships, one in each direction, that carry the line name. If the stations are already there, loading is skipped unless `RELOAD_DATA = True`.

In [6]:
def clear_database(session):
    # Drop all constraints
    constraints = session.run("SHOW CONSTRAINTS YIELD name RETURN name").data()
    for c in constraints:
        session.run(f"DROP CONSTRAINT {c['name']} IF EXISTS")

    # Drop all non-lookup indexes
    indexes = session.run("""
        SHOW INDEXES YIELD name, type
        WHERE type <> 'LOOKUP'
        RETURN name
    """).data()
    for idx in indexes:
        session.run(f"DROP INDEX {idx['name']} IF EXISTS")

    # Delete all nodes and relationships in batches
    while True:
        result = session.run("""
            MATCH (n) WITH n LIMIT 1000
            DETACH DELETE n
            RETURN count(*) AS deleted
        """)
        if result.single()["deleted"] == 0:
            break

    print("Database cleared.")

if CLEAR_DATABASE:
    try:
        with open_session() as session:
            clear_database(session)
    except Exception as exc:
        problem(exc)
else:
    print("CLEAR_DATABASE is False, so the database is left as it is.")

Database cleared.


In [7]:
try:
    stations_df    = pd.read_csv("../data/london_stations.csv")
    connections_df = pd.read_csv("../data/london_connections.csv")
    lines_df       = pd.read_csv("../data/london_lines.csv")
except FileNotFoundError as exc:
    stations_df = connections_df = lines_df = None
    print(f"Can't find {exc.filename}. Put the three CSV files in the shared data folder, one level above this notebook's folder (see the prerequisites).")
    if DEBUG:
        raise
else:
    # Strip whitespace from column names and string values
    for df in [stations_df, connections_df, lines_df]:
        df.columns = df.columns.str.strip()
        df.update(df.select_dtypes(include="str").apply(lambda c: c.str.strip()))

    # Merge line colors onto connections
    connections_df = connections_df.merge(lines_df, on="Tube Line", how="inner")

    # Reduce to connected stations only
    connected_stations = set(connections_df["From Station"]) | set(connections_df["To Station"])
    stations_df = stations_df[stations_df["Station"].isin(connected_stations)].copy()

    print(f"Stations:    {len(stations_df)}")
    print(f"Connections: {len(connections_df)}")

Stations:    349
Connections: 458


In [8]:
def ingest(driver, stations_df, connections_df):
    with open_session() as session:

        session.run("""
            CREATE CONSTRAINT station_name IF NOT EXISTS
            FOR (s:Station) REQUIRE s.name IS UNIQUE
        """)

        station_records = (
            stations_df
            .rename(columns={
                "Station":   "name",
                "Latitude":  "lat",
                "Longitude": "lon",
                "Zone":      "zone",
                "Postcode":  "postcode"
            })
            [["name", "lat", "lon", "zone", "postcode"]]
            .to_dict("records")
        )

        session.run("""
            UNWIND $rows AS row
            MERGE (s:Station {name: row.name})
            SET s.lat      = toFloat(row.lat),
                s.lon      = toFloat(row.lon),
                s.zone     = row.zone,
                s.postcode = row.postcode
        """, rows=station_records)
        print(f"Loaded {len(station_records)} stations.")

        conn_records = (
            connections_df
            .rename(columns={
                "From Station": "from_name",
                "To Station":   "to_name",
                "Tube Line":    "line",
                "Color":        "color"
            })
            [["from_name", "to_name", "line", "color"]]
            .to_dict("records")
        )

        session.run("""
            UNWIND $rows AS row
            MATCH (a:Station {name: row.from_name})
            MATCH (b:Station {name: row.to_name})
            MERGE (a)-[:CONNECTS_TO {line: row.line, color: row.color}]->(b)
            MERGE (b)-[:CONNECTS_TO {line: row.line, color: row.color}]->(a)
        """, rows=conn_records)
        print(f"Loaded {len(conn_records)} connections (bidirectional).")

try:
    with open_session() as session:
        existing = session.run("MATCH (s:Station) RETURN count(s) AS n").single()["n"]

    if existing and not RELOAD_DATA:
        print(f"{existing} stations are already loaded. Set RELOAD_DATA = True to load them again.")
    elif stations_df is None:
        print("The CSV files weren't loaded, so there is nothing to ingest. Fix the previous cell first.")
    else:
        if existing:
            with open_session() as session:
                session.run("MATCH (s:Station) DETACH DELETE s")
            print("Station nodes deleted.")
        ingest(driver, stations_df, connections_df)
except Exception as exc:
    problem(exc)

Loaded 349 stations.
Loaded 458 connections (bidirectional).


In [9]:
try:
    with open_session() as session:
        counts = session.run("""
            MATCH (s:Station)
            WITH count(s) AS stations
            OPTIONAL MATCH ()-[r:CONNECTS_TO]->()
            RETURN stations, count(r) AS connections
        """).single()
    print(f"Stations: {counts['stations']}, connections: {counts['connections']}")
except Exception as exc:
    problem(exc)

Stations: 349, connections: 916


## 6. Helpers

`run_cypher` runs one statement. A single map prints as JSON, anything else shows as a table.

The rest is the whole pipeline in small pieces:

- `project_network` and `propose_routes` ask the graph for the candidate routes, with the lines and interchanges of each.
- `STATUS_QUERY` reads the live status of every station on each route and says it in words.
- `build_inputs` turns routes and status into the state and the question that go to Jev.
- `DECIDE_QUERY` is the Jev call itself.
- `remember` stores the decision in the graph.

In [10]:
def run_cypher(query, **params):
    """Run one statement and show the result. A single map prints as JSON, anything
    else as a table. Statements that change the graph also report what changed."""
    try:
        with open_session() as session:
            result = session.run(query, **params)
            records = [r.data() for r in result]
            summary = result.consume()
    except Exception as exc:
        problem(exc)
        return

    if len(records) == 1 and len(records[0]) == 1 and isinstance(next(iter(records[0].values())), dict):
        print(json.dumps(next(iter(records[0].values())), indent=2, default=str))
    elif records:
        display(pd.DataFrame(records))
    else:
        print("No results.")

    changes = {name: getattr(summary.counters, name, 0) for name in (
        "nodes_created", "nodes_deleted", "relationships_created",
        "relationships_deleted", "properties_set")}
    changes = {k: v for k, v in changes.items() if v}
    if changes:
        print("Changes:", ", ".join(f"{k} {v}" for k, v in changes.items()))
    ms = (summary.result_available_after or 0) + (summary.result_consumed_after or 0)
    print(f"Completed after {ms:,} ms")

In [11]:
# 1. The graph proposes: project the network, ask for a pool of short routes, keep the distinct ones.

PROJECT_QUERY = """
MATCH (a:Station)-[r:CONNECTS_TO]->(b:Station)
WHERE $lines IS NULL OR r.line IN $lines
WITH DISTINCT a, b
WITH gds.graph.project($name, a, b) AS g
RETURN g.nodeCount AS stations, g.relationshipCount AS connections
"""

YENS_QUERY = """
MATCH (source:Station {name: $origin}), (target:Station {name: $destination})
CALL gds.shortestPath.yens.stream($name, {sourceNode: source, targetNode: target, k: $pool})
YIELD index, nodeIds
RETURN index, [id IN nodeIds | gds.util.asNode(id).name] AS stations
ORDER BY index
"""

HOP_LINES_QUERY = """
UNWIND range(0, size($routes) - 1) AS ri
UNWIND range(0, size($routes[ri]) - 2) AS i
MATCH (a:Station {name: $routes[ri][i]})-[r:CONNECTS_TO]->(b:Station {name: $routes[ri][i + 1]})
WHERE $lines IS NULL OR r.line IN $lines
RETURN ri, i, collect(DISTINCT r.line) AS lines
ORDER BY ri, i
"""

def project_network(lines):
    """Put the stations and connections into a GDS projection, one connection per pair of stations."""
    with open_session() as session:
        session.run("CALL gds.graph.drop($name, false)", name=GRAPH_NAME).consume()
        return session.run(PROJECT_QUERY, name=GRAPH_NAME, lines=lines).single().data()

def line_plan(hop_lines):
    """Pick one line for each hop so the passenger changes as seldom as possible.
    Returns the legs of the journey as {line, first, last} hop numbers."""
    best = [{line: (0, None) for line in hop_lines[0]}]
    for lines_here in hop_lines[1:]:
        best.append({line: min((changes + (0 if prev == line else 1), prev)
                               for prev, (changes, _) in best[-1].items())
                     for line in lines_here})
    line = min(best[-1], key=lambda name: best[-1][name][0])
    chosen = [line]
    for step in range(len(best) - 1, 0, -1):
        line = best[step][line][1]
        chosen.append(line)
    chosen.reverse()
    legs = []
    for hop, line in enumerate(chosen):
        if legs and legs[-1]["line"] == line:
            legs[-1]["last"] = hop
        else:
            legs.append({"line": line, "first": hop, "last": hop})
    return legs

def describe(route):
    """The words Jev gets for one option: the stations, the stops and how the lines connect."""
    stations, legs = route["stations"], route["legs"]
    if len(legs) == 1:
        how = f"On the {legs[0]['line']} line."
    else:
        parts = [f"{leg['line']} line to {stations[leg['last'] + 1]}" for leg in legs]
        how = f"Changes: {', '.join(parts)} ({len(legs) - 1} interchange{'s' if len(legs) > 2 else ''})."
    return f"Via {', '.join(stations[1:-1])}. {route['stops']} stops. {how}"

def choose_routes(pool, k):
    """Keep the k cheapest routes that are really different from each other."""
    chosen = []
    for route in sorted(pool, key=lambda r: (r["cost"], r["stops"])):
        middle = set(route["stations"][1:-1])
        distinct = True
        for other in chosen:
            theirs = set(other["stations"][1:-1])
            if len(middle & theirs) / (len(middle | theirs) or 1) > MAX_OVERLAP:
                distinct = False
        if distinct:
            chosen.append(route)
        if len(chosen) == k:
            break
    return chosen

def label_routes(routes):
    """Name each route after the first station that only it uses, so the options read 'Via Waterloo'."""
    for route in routes:
        others = set()
        for other in routes:
            if other is not route:
                others |= set(other["stations"])
        unique = [s for s in route["stations"][1:-1] if s not in others]
        route["label"] = f"Via {unique[0]}" if unique else f"Route {route['index'] + 1}"
        route["key"] = re.sub(r"[^a-z0-9]+", "_", route["label"].lower()).strip("_")

def propose_routes(origin, destination, k, lines=None, pool=None):
    pool = POOL if pool is None else pool
    with open_session() as session:
        found = [r.data()["stations"] for r in session.run(
            YENS_QUERY, name=GRAPH_NAME, origin=origin, destination=destination, pool=pool)]
        hops = {}
        if found:
            for r in session.run(HOP_LINES_QUERY, routes=found, lines=lines):
                hops.setdefault(r["ri"], {})[r["i"]] = r["lines"]
    candidates = []
    for ri, stations in enumerate(found):
        by_hop = hops.get(ri, {})
        if len(by_hop) != len(stations) - 1 or any(not l for l in by_hop.values()):
            continue
        legs = line_plan([by_hop[i] for i in range(len(stations) - 1)])
        stops, interchanges = len(stations) - 1, len(legs) - 1
        candidates.append({"stations": stations, "stops": stops, "legs": legs, "interchanges": interchanges,
                           "cost": stops + INTERCHANGE_COST * interchanges})
    routes = choose_routes(candidates, k)
    for index, route in enumerate(routes):
        route["index"] = index
        route["description"] = describe(route)
    label_routes(routes)
    return routes

def routes_table(routes):
    return pd.DataFrame([{"route": r["label"], "stops": r["stops"], "interchanges": r["interchanges"],
                          "lines": " > ".join(leg["line"] for leg in r["legs"]),
                          "stations": " > ".join(r["stations"][1:-1])} for r in routes])

print("Route helpers defined.")

Route helpers defined.


In [12]:
# 2. Cypher adds the live status of every route, in words.

STATUS_QUERY = """
UNWIND $routes AS route
UNWIND range(0, size(route.stops) - 1) AS i
MATCH (s:Station {name: route.stops[i]})
WITH route.label AS label, i, s ORDER BY label, i
WITH label, collect(s) AS stops
WITH label,
     [s IN stops WHERE NOT coalesce(s.active, true) | s.name] AS suspended,
     [s IN stops WHERE coalesce(s.active, true) AND coalesce(s.delay_minutes, 0) > 0 | s] AS late
WITH label, suspended,
     [s IN late | s.name] AS late_names,
     [s IN late | s.name + ' (' + toString(s.delay_minutes) + ' min)'] AS delayed
WITH label, suspended, late_names,
     CASE WHEN size(suspended) = 0 THEN ''
          ELSE 'Suspended: ' + reduce(t = head(suspended), n IN tail(suspended) | t + ', ' + n) + '. '
     END AS off_text,
     CASE WHEN size(delayed) = 0 THEN ''
          ELSE 'Delays: ' + reduce(t = head(delayed), n IN tail(delayed) | t + ', ' + n) + '.'
     END AS late_text
RETURN label, suspended, late_names,
       CASE WHEN off_text + late_text = '' THEN 'All stations clear.'
            ELSE trim(off_text + late_text)
       END AS status
"""

def route_status(routes):
    """Returns {label: {status, suspended, late_names}} for the routes, read from the graph now."""
    with open_session() as session:
        rows = [r.data() for r in session.run(
            STATUS_QUERY, routes=[{"label": r["label"], "stops": r["stations"]} for r in routes])]
    return {r["label"]: r for r in rows}

print("Status query defined.")

Status query defined.


In [13]:
# 3. Jev decides.

DECIDE_QUERY = """
RETURN jev.decide($state, $questions, $options) AS result
"""

def build_inputs(routes, lines, status):
    """The state and the question for Jev, from the routes the graph proposed and their live status."""
    network = f"London Underground, {lines[0]} line" if lines and len(lines) == 1 else "London Underground"
    state = {"current_station": ORIGIN, "destination": DESTINATION, "network": network}
    criteria = {}
    for r in routes:
        state["status_" + r["key"]] = status[r["label"]]["status"]
        criteria[r["label"]] = r["description"]
    questions = {"route": {
        "type": "choice",
        "instructions": (f"Which route should the passenger take from {ORIGIN} to reach {DESTINATION} "
                         "as reliably as possible given the current state of the routes?"),
        "criteria": criteria,
    }}
    return state, questions

def ask_jev(state, questions, options):
    with open_session() as session:
        record = session.run(DECIDE_QUERY, state=state, questions=questions, options=options).single()
    return record["result"] if record else None

def margin_of(result):
    if not result or result.get("error_message"):
        return None
    probs = sorted(result["answers"]["route"]["probabilities"].values(), reverse=True)
    return probs[0] - probs[1] if len(probs) > 1 else None

def show(result):
    if result is None:
        print("No result.")
        return
    if result["error_message"]:
        print(f"Error: {describe_error(result['error_message'])}")
        return
    answer = result["answers"]["route"]
    probs  = answer["probabilities"]
    split  = ", ".join(f"{k} {v:.2f}" for k, v in sorted(probs.items(), key=lambda kv: -kv[1]))
    meta   = result["metadata"]
    print(f"{answer['choice']} | {split} | margin {margin_of(result):.2f} | "
          f"{meta['latency_ms']} ms | {meta.get('model')}")

print("Decision helpers defined.")

Decision helpers defined.


In [14]:
# 4. The graph remembers.

STORE_QUERY = """
MATCH (a:Station {name: $origin}), (b:Station {name: $destination})
CREATE (d:Decision {id: randomUUID(), asked_at: datetime(), choice: $choice,
                    margin: $margin, probability: $probability, model: $model,
                    route: $route, state: $state_text})
CREATE (d)-[:FROM]->(a)
CREATE (d)-[:TO]->(b)
WITH d
OPTIONAL MATCH (s:Station) WHERE s.name IN $disrupted
FOREACH (x IN CASE WHEN s IS NULL THEN [] ELSE [s] END | CREATE (d)-[:CONCERNS]->(x))
RETURN DISTINCT d.id AS id
"""

def remember(result, routes, state, status):
    """Store the decision as a node, linked to the stations that were disrupted."""
    answer    = result["answers"]["route"]
    chosen    = next((r for r in routes if r["label"] == answer["choice"]), None)
    disrupted = sorted({n for s in status.values() for n in s["suspended"] + s["late_names"]})
    with open_session() as session:
        record = session.run(
            STORE_QUERY, origin=ORIGIN, destination=DESTINATION, choice=answer["choice"],
            margin=margin_of(result), probability=answer["probabilities"][answer["choice"]],
            model=result["metadata"].get("model"), route=chosen["stations"] if chosen else [],
            state_text=json.dumps(state), disrupted=disrupted).single()
    return record["id"]

def decide_now(routes, lines=None, options=None, store=True, quiet=False):
    """Read the live status, ask Jev, remember the decision. Returns the result map, or None after printing why not."""
    options = HOSTED_OPTIONS if options is None else options
    if not routes:
        print("No routes to choose between. Run section 7 first.")
        return None
    try:
        status = route_status(routes)
        state, questions = build_inputs(routes, lines, status)
        result = ask_jev(state, questions, options)
        if not quiet:
            show(result)
        if store and result and not result["error_message"]:
            print(f"Stored decision {remember(result, routes, state, status)[:8]}.")
    except Exception as exc:
        problem(exc)
        return None
    return result

print("Memory helpers defined.")

Memory helpers defined.


### Pre-flight check

Run this before the talk, with the same network you'll use on the day. It projects the network, asks the graph for the routes, reads their status and asks Jev once through each service. Nothing is stored and no disruption is changed.

In [15]:
def preflight():
    try:
        if driver is None:
            raise NotConnected()
        driver.verify_connectivity()
        with open_session() as session:
            functions = [r["name"] for r in session.run(
                "SHOW FUNCTIONS YIELD name WHERE name STARTS WITH 'jev' RETURN name")]
            gds_ready = session.run(
                "SHOW PROCEDURES YIELD name WHERE name = 'gds.shortestPath.yens.stream' RETURN count(*) AS n").single()["n"]
            stations  = session.run("MATCH (s:Station) RETURN count(s) AS n").single()["n"]
            leftover  = session.run(
                "MATCH (s:Station) WHERE s.active IS NOT NULL OR s.delay_minutes IS NOT NULL "
                "RETURN count(s) AS n").single()["n"]
    except Exception as exc:
        print(f"Neo4j:       NOT READY. {explain(exc)}")
        if DEBUG:
            raise
        return
    print("Neo4j:       connected")
    print(f"jev.decide:  {'registered' if 'jev.decide' in functions else 'NOT READY (check the JAR, the allowlist and a restart)'}")
    print(f"GDS (Yen's): {'registered' if gds_ready else 'NOT READY (install GDS, add gds.* to the allowlist and unrestricted lines, restart)'}")
    print(f"Stations:    {stations if stations else 'NOT READY (run section 5)'}")
    print("Disruptions: " + ("none" if not leftover else f"{leftover} station(s) still have one (run the reset in section 11)"))
    if not (stations and gds_ready and 'jev.decide' in functions):
        return
    try:
        project_network(LINES)
        routes = propose_routes(ORIGIN, DESTINATION, K, LINES)
    except Exception as exc:
        print(f"Routes:      NOT READY. {explain(exc)}")
        if DEBUG:
            raise
        return
    print(f"Routes:      {len(routes)} proposed ({', '.join(r['label'] for r in routes)})")
    for label, options in (("hosted", HOSTED_OPTIONS), ("local", LOCAL_OPTIONS)):
        try:
            state, questions = build_inputs(routes, LINES, route_status(routes))
            result = ask_jev(state, questions, options)
        except Exception as exc:
            print(f"Decision, {label}: NOT READY. {explain(exc)}")
            if DEBUG:
                raise
            continue
        if result is None or result["error_message"]:
            print(f"Decision, {label}: NOT READY. {describe_error(result['error_message']) if result else 'no result'}")
        else:
            print(f"Decision, {label}: ready, {result['answers']['route']['choice']}, {result['metadata']['latency_ms']} ms")

print("Pre-flight defined.")

Pre-flight defined.


In [16]:
preflight()

Neo4j:       connected
jev.decide:  registered
GDS (Yen's): registered
Stations:    349
Disruptions: none
Routes:      2 proposed (Via Waterloo, Via Elephant and Castle)
Decision, hosted: ready, Via Waterloo, 271 ms
Decision, local: ready, Via Waterloo, 3323 ms


## 7. The graph proposes the routes

First project the network into GDS. With `LINES = ["Northern"]` only Northern line connections go in, one per pair of stations. Then ask Yen's algorithm for a pool of short routes from Kennington to Euston, work out which line each hop uses (and how many interchanges that needs), and keep the `K` best routes that are really different from each other. On the Northern line that gives the two branches.

The routes are found from the layout of the network alone. Disruptions are not considered here, because the next step reads them and Jev weighs them.

In [17]:
try:
    print(project_network(LINES))
except Exception as exc:
    problem(exc)

{'stations': 52, 'connections': 106}


In [18]:
try:
    ROUTES = propose_routes(ORIGIN, DESTINATION, K, LINES)
    if ROUTES:
        display(routes_table(ROUTES))
    else:
        print("No routes found. Check the station names and the lines.")
except Exception as exc:
    ROUTES = []
    problem(exc)

,route,stops,interchanges,lines,stations
0,Via Waterloo,8,0,Northern,Waterloo > Embankment > Charing Cross > Leices...
1,Via Elephant and Castle,9,0,Northern,Elephant and Castle > Borough > London Bridge ...


## 8. The live state of each route

No station has an `active` or `delay_minutes` property yet, so everything counts as running normally. This statement reads those properties for every station on each route and writes the status in words, the same words Jev will see.

In [19]:
print(STATUS_QUERY)


UNWIND $routes AS route
UNWIND range(0, size(route.stops) - 1) AS i
MATCH (s:Station {name: route.stops[i]})
WITH route.label AS label, i, s ORDER BY label, i
WITH label, collect(s) AS stops
WITH label,
     [s IN stops WHERE NOT coalesce(s.active, true) | s.name] AS suspended,
     [s IN stops WHERE coalesce(s.active, true) AND coalesce(s.delay_minutes, 0) > 0 | s] AS late
WITH label, suspended,
     [s IN late | s.name] AS late_names,
     [s IN late | s.name + ' (' + toString(s.delay_minutes) + ' min)'] AS delayed
WITH label, suspended, late_names,
     CASE WHEN size(suspended) = 0 THEN ''
          ELSE 'Suspended: ' + reduce(t = head(suspended), n IN tail(suspended) | t + ', ' + n) + '. '
     END AS off_text,
     CASE WHEN size(delayed) = 0 THEN ''
          ELSE 'Delays: ' + reduce(t = head(delayed), n IN tail(delayed) | t + ', ' + n) + '.'
     END AS late_text
RETURN label, suspended, late_names,
       CASE WHEN off_text + late_text = '' THEN 'All stations clear.'
        

In [20]:
try:
    status = route_status(ROUTES)
    display(pd.DataFrame([{"route": r["label"], "status": status[r["label"]]["status"]} for r in ROUTES]))
except Exception as exc:
    problem(exc)

,route,status
0,Via Waterloo,All stations clear.
1,Via Elephant and Castle,All stations clear.


## 9. Jev decides

Jev gets the state and a question with one option per route. The call is a single line of Cypher. The decision is stored in the graph as it comes back.

In [21]:
print(DECIDE_QUERY)


RETURN jev.decide($state, $questions, $options) AS result



In [22]:
result = decide_now(ROUTES, LINES)
if result and not result["error_message"]:
    print()
    print(json.dumps(result, indent=2, default=str))

Via Waterloo | Via Waterloo 0.54, Via Elephant and Castle 0.46 | margin 0.08 | 145 ms | jev-1.13.0
Stored decision 76e26633.

{
  "answers": {
    "route": {
      "type": "choice",
      "choice": "Via Waterloo",
      "probabilities": {
        "Via Waterloo": 0.54,
        "Via Elephant and Castle": 0.46
      },
      "confidence": 0.08
    }
  },
  "error_message": null,
  "metadata": {
    "authenticated": true,
    "model": "jev-1.13.0",
    "latency_ms": 145,
    "attempts": 1
  }
}


## 10. Take a station offline

Suspend Warren Street, which sits on one of the routes, then run the same two steps again. Nothing about the call changes, only the graph.

In [23]:
run_cypher("""
MATCH (s:Station {name: 'Warren Street'})
SET s.active = false
RETURN s.name AS station, s.active AS active
""")

,station,active
0,Warren Street,False


Changes: properties_set 1
Completed after 31 ms


In [24]:
try:
    status = route_status(ROUTES)
    display(pd.DataFrame([{"route": r["label"], "status": status[r["label"]]["status"]} for r in ROUTES]))
except Exception as exc:
    problem(exc)
result = decide_now(ROUTES, LINES)

,route,status
0,Via Waterloo,Suspended: Warren Street.
1,Via Elephant and Castle,All stations clear.


Via Elephant and Castle | Via Elephant and Castle 1.00, Via Waterloo 0.00 | margin 1.00 | 150 ms | jev-1.13.0
Stored decision 81b9e05f.


### Your turn

Reset the disruption, delay Goodge Street by 5 minutes and ask again. A delay is a softer signal than a suspension, so watch the margin.

In [25]:
run_cypher("""
MATCH (s:Station)
WHERE s.active IS NOT NULL OR s.delay_minutes IS NOT NULL
REMOVE s.active, s.delay_minutes
""")
run_cypher("""
MATCH (s:Station {name: 'Goodge Street'})
SET s.delay_minutes = 5
RETURN s.name AS station, s.delay_minutes AS delay_minutes
""")
result = decide_now(ROUTES, LINES)

No results.
Changes: properties_set 1
Completed after 28 ms


,station,delay_minutes
0,Goodge Street,5


Changes: properties_set 1
Completed after 24 ms
Via Elephant and Castle | Via Elephant and Castle 0.99, Via Waterloo 0.01 | margin 0.98 | 142 ms | jev-1.13.0
Stored decision 62f26ba8.


## 11. The graph remembers

Every decision is a `Decision` node linked to its origin and destination and to the stations that were disrupted at the time. So the decisions are data we can query.

Which decisions involved Warren Street?

In [26]:
run_cypher("""
MATCH (d:Decision)-[:CONCERNS]->(:Station {name: 'Warren Street'})
RETURN toString(datetime.truncate('second', d.asked_at)) AS asked, d.choice AS choice, round(d.margin, 2) AS margin
ORDER BY asked
""")

,asked,choice,margin
0,2026-11-12T04:46:35Z,Via Elephant and Castle,1.0


Completed after 63 ms


Which decisions were close calls? A low margin is a decision worth a human check, and the query shows what was disrupted when it was made.

In [27]:
run_cypher("""
MATCH (d:Decision)
WHERE d.margin < 0.6
OPTIONAL MATCH (d)-[:CONCERNS]->(s:Station)
RETURN toString(datetime.truncate('second', d.asked_at)) AS asked, d.choice AS choice, round(d.margin, 2) AS margin,
       collect(s.name) AS disrupted
ORDER BY margin
""")

,asked,choice,margin,disrupted
0,2026-11-12T04:46:35Z,Via Waterloo,0.08,[]


Completed after 73 ms


Everything stored, newest first.

In [28]:
run_cypher("""
MATCH (d:Decision)-[:FROM]->(a:Station), (d)-[:TO]->(b:Station)
OPTIONAL MATCH (d)-[:CONCERNS]->(s:Station)
RETURN toString(datetime.truncate('second', d.asked_at)) AS asked, a.name + ' to ' + b.name AS trip, d.choice AS choice,
       round(d.margin, 2) AS margin, collect(s.name) AS disrupted
ORDER BY asked DESC
""")

,asked,trip,choice,margin,disrupted
0,2026-11-12T04:46:35Z,Kennington to Euston,Via Waterloo,0.08,[]
1,2026-11-12T04:46:35Z,Kennington to Euston,Via Elephant and Castle,1.00,[Warren Street]
2,2026-11-12T04:46:35Z,Kennington to Euston,Via Elephant and Castle,0.98,[Goodge Street]


Completed after 78 ms


### Reset

Remove the disruption properties, so every station counts as running normally again. Then delete the stored decisions. Run both between rehearsals. The stations and connections stay.

In [29]:
run_cypher("""
MATCH (s:Station)
WHERE s.active IS NOT NULL OR s.delay_minutes IS NOT NULL
REMOVE s.active, s.delay_minutes
""")

No results.
Changes: properties_set 1
Completed after 3 ms


In [30]:
run_cypher("""
MATCH (d:Decision)
DETACH DELETE d
""")

No results.
Changes: nodes_deleted 3, relationships_deleted 8
Completed after 28 ms


## 12. Optional: the same call on a local model

Only the options change. The first call after the model loads is slow, so warm it up before the talk. The decision isn't stored, so the graph keeps only hosted decisions.

In [31]:
result = decide_now(ROUTES, LINES, options=LOCAL_OPTIONS, store=False)

Via Waterloo | Via Waterloo 0.50, Via Elephant and Castle 0.50 | margin 0.01 | 11 ms | tev1:4b


## 13. Optional: open the network to every line

Now let the graph use any line and propose three routes. Interchanges become part of the answer, which is where a graph earns its keep over a hardcoded list. The routes are ranked by stops plus a penalty for each interchange, and a route is only kept if it shares at most half its stations with the ones already chosen, so the options are genuinely different. Each option tells Jev which lines it uses and where it changes.

This reprojects the network, asks Jev without storing the decision, and then puts the projection back as it was.

In [32]:
def explore(lines=None, k=3):
    try:
        project_network(lines)
        routes = propose_routes(ORIGIN, DESTINATION, k, lines)
        if routes:
            display(routes_table(routes))
            decide_now(routes, lines, store=False)
        else:
            print("No routes found.")
    except Exception as exc:
        problem(exc)
    finally:
        try:
            project_network(LINES)
        except Exception:
            pass

explore(lines=None, k=3)

,route,stops,interchanges,lines,stations
0,Via Embankment,8,0,Northern,Waterloo > Embankment > Charing Cross > Leices...
1,Via Elephant and Castle,9,0,Northern,Elephant and Castle > Borough > London Bridge ...
2,Via Westminster,6,2,Northern > Jubilee > Victoria,Waterloo > Westminster > Green Park > Oxford C...


Via Elephant and Castle | Via Elephant and Castle 0.62, Via Embankment 0.30, Via Westminster 0.08 | margin 0.32 | 205 ms | jev-1.13.0


## 14. Teardown

In [33]:
try:
    with open_session() as session:
        session.run("CALL gds.graph.drop($name, false)", name=GRAPH_NAME).consume()
except Exception as exc:
    problem(exc)

if driver is not None:
    driver.close()
print("Connection closed.")

Connection closed.
